In [2]:
import pandas as pd
import numpy as np
import json
import os
import time
from openai import OpenAI
from dotenv import load_dotenv
import warnings

warnings.filterwarnings('ignore')
load_dotenv()

print("=" * 80)
print("Step 6: Agent #2 — Consulting Report Generator (Full Run)")
print("=" * 80)

# ============================================================================
# 1. Feature Name Mapping Dictionary
# ============================================================================
FEATURE_MAPPING = {
    'FN1_1': 'Current Assets', 'FN1_2': 'Non-Current Assets', 'FN1_3': 'Quick Assets',
    'FN1_4': 'Inventory', 'FN1_5': 'Tangible Assets', 'FN1_6': 'Work-in-Progress',
    'FN1_7': 'Cash', 'FN1_8': 'Cash Equivalents', 'FN1_9': 'Marketable Securities',
    'FN1_10': 'Cash and Cash Equivalents', 'FN1_11': 'Accounts Receivable',
    'FN1_11_1': 'Prior-Year Accounts Receivable', 'FN1_11_2': 'AR Disposal Loss',
    'FN1_11_3': 'Intangible Assets', 'FN1_11_4': 'Investment Assets',
    'FN1_13': 'Total Assets', 'FN1_13_1': 'Prior-Year Total Assets',
    'FN1_14': 'Current Liabilities', 'FN1_15': 'Short-Term Borrowings',
    'FN1_16': 'Borrowings', 'FN1_17': 'Accounts Payable',
    'FN1_18': 'Non-Current Liabilities', 'FN1_19': 'Total Liabilities',
    'FN1_20': 'Equity', 'FN1_21': 'Capital Surplus', 'FN1_22': 'Retained Earnings',
    'FN1_23': 'Reserves', 'FN1_24': 'Total Equity',
    'FN2_1': 'Revenue', 'FN2_1_1': 'Prior-Year Revenue',
    'FN2_2': 'Cost of Goods Sold', 'FN2_2_1': 'Gross Profit',
    'FN2_3': 'SG&A Expenses', 'FN2_4': 'Finance Costs',
    'FN2_5': 'Operating Income', 'FN2_5_1': 'Prior-Year Operating Income',
    'FN2_7': 'Non-Operating Income', 'FN2_8': 'Non-Operating Expenses',
    'FN2_9': 'Pre-Tax Income', 'FN2_10': 'Net Income',
    'FN2_10_1': 'Prior-Year Net Income',
    'FN3_1': 'Cash Flow', 'FN3_2': 'Operating Cash Flow',
    'FN3_2_1': 'Investing Cash Flow', 'FN3_3': 'Debt Service Coverage Ratio',
    'FN3_6': 'Reserve Ratio', 'FN3_7': 'EBIT', 'FN3_8': 'EBITDA',
    'FN3_10': 'Liquidation Value Ratio', 'FN3_11': 'Net Working Capital',
    'FN3_11_1': 'Net Debt', 'PERF_12M': 'Bankruptcy Flag'
}

# ============================================================================
# 2. Data Loading
# ============================================================================
try:
    df_input = pd.read_csv('agent1_interpretation_results.csv')
    print(f"Data loaded: {len(df_input)} records")
except FileNotFoundError:
    raise FileNotFoundError("Input file from previous step (agent1_interpretation_results.csv) not found.")

# ============================================================================
# 3. Report Generator Class (Logic-Reinforced Version)
# ============================================================================
class ReportGenerator:
    def __init__(self):
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model = os.getenv('LLM_MODEL', 'gpt-4o-mini')

    def _format_financial_guide(self, row):
        """Generate numerical change guide with directional interpretation hints."""
        changes_list = []
        features = [col.replace('Original_', '') for col in row.index if col.startswith('Original_')]

        for feat in features:
            try:
                original = float(row[f'Original_{feat}'])
                target = float(row[f'CF_{feat}'])
                change = float(row[f'Change_{feat}'])

                if abs(change) > 0.001:
                    pct_change = (change / abs(original)) * 100 if original != 0 else 0
                    direction = "increase" if change > 0 else "decrease"
                    feat_name = FEATURE_MAPPING.get(feat, feat)

                    # [Logic Guardrail] Directional interpretation hints per variable type
                    hint = ""
                    if "Borrowings" in feat_name or "Liabilities" in feat_name:
                        if direction == "increase":
                            hint = " -> (Note: This is a strategic borrowing expansion to secure liquidity/working capital. Do NOT recommend 'debt repayment'.)"
                        else:
                            hint = " -> (Debt repayment to strengthen financial soundness)"
                    elif "Assets" in feat_name or "Inventory" in feat_name:
                        if direction == "decrease":
                            hint = " -> (Disposal of idle assets or inventory liquidation)"
                        else:
                            hint = " -> (Asset expansion to increase firm scale)"
                    elif "Accounts Receivable" in feat_name:
                        if direction == "decrease":
                            hint = " -> (Active AR collection to convert to cash)"

                    desc = f"- **{feat_name}**: {original:,.0f} → {target:,.0f} ({abs(pct_change):.1f}% {direction}){hint}"
                    changes_list.append((abs(pct_change), desc))
            except Exception:
                continue

        # Sort by magnitude, keep top 10
        changes_list.sort(key=lambda x: x[0], reverse=True)
        return "\n".join([item[1] for item in changes_list[:10]])

    def generate_report(self, row):
        company_id = row['ID']
        financial_guide_text = self._format_financial_guide(row)
        prob_improvement = (row['Original_Proba'] - row['Target_Proba']) * 100

        system_prompt = """
        You are a corporate credit insurance underwriting specialist with 20 years of experience.
        Interpret the direction of numerical increases and decreases accurately
        and produce a logically consistent financial improvement report.

        [Absolute Rules]
        1. **Data-driven interpretation:** If the data indicates a 'liability increase',
           interpret it as 'securing funding' — never write 'debt repayment'.
        2. **Terminology:** Use standard financial terms instead of variable codes (FN...).
        3. **Verbatim figures:** Every CF target value from the guide MUST appear
           in the report body EXACTLY as given — same integer, same unit.
           Do NOT round, approximate, or paraphrase any number.
        4. **Percentage changes:** When stating a percentage change, compute it as
           (Target - Original) / |Original| × 100 and round to one decimal place.
           Show your arithmetic: write "(X → Y, +Z.Z%)".
        5. **No invented numbers:** Never introduce any figure that does not appear
           in the input data. If a value is unknown, omit it rather than estimate.
        """

        user_prompt = f"""
        [Company Information]
        - ID: {company_id}
        - Bankruptcy probability improvement target: {row['Original_Proba']:.1%} → {row['Target_Proba']:.1%} ({prob_improvement:.1f}pp improvement)

        [AI Analysis Summary]
        - Strategy: {row.get('selection_reason', '-')}
        - Feasibility: {row.get('feasibility_assessment', '-')}

        [Financial Indicator Change Guide (figures and directional hints)]
        {financial_guide_text}

        [Report Structure (Markdown)]
        # 1. Executive Summary (3-sentence overview)
        # 2. Current Status Analysis (diagnosis)
        # 3. Improvement Scenarios (detailed targets — cite figures)
        # 4. Action Roadmap (short-term / medium-term)
        # 5. Risk Management
        # 6. Performance KPIs (3 indicators)
        """

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.5
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"

# ============================================================================
# 4. Full Execution
# ============================================================================
generator = ReportGenerator()
generated_reports = []

# [Config] TEST_MODE = False (full run)
TEST_MODE = False

print(f"\n[FULL MODE] Generating reports for all {len(df_input)} firms.")
print("Running... (estimated time: 10–20 minutes)")

try:
    from tqdm import tqdm
    iterator = tqdm(df_input.iterrows(), total=len(df_input), desc="Generating Reports")
except ImportError:
    iterator = df_input.iterrows()

for idx, row in iterator:
    report_text = generator.generate_report(row)
    generated_reports.append({
        'company_id': row['ID'],
        'original_proba': row['Original_Proba'],
        'target_proba': row['Target_Proba'],
        'report_content': report_text
    })

# ============================================================================
# 5. Save Outputs
# ============================================================================
output_csv = 'agent2_consulting_reports.csv'
df_reports = pd.DataFrame(generated_reports)
df_reports.to_csv(output_csv, index=False, encoding='utf-8-sig')

print("\n" + "=" * 80)
print(f"Generation complete: {len(df_reports)} reports saved")
print(f"Combined output file: {output_csv}")
print("=" * 80)

# Save individual Markdown files (reports/ folder)
output_dir = 'reports'
os.makedirs(output_dir, exist_ok=True)
for _, row in df_reports.iterrows():
    with open(f"{output_dir}/Report_{int(row['company_id'])}.md", 'w', encoding='utf-8') as f:
        f.write(row['report_content'])

print(f"Individual reports saved in: {output_dir}/")

Step 6: Agent #2 — Consulting Report Generator (Full Run)
Data loaded: 266 records

[FULL MODE] Generating reports for all 266 firms.
Running... (estimated time: 10–20 minutes)


Generating Reports: 100%|████████████████████████████████████████████████████████████| 266/266 [51:28<00:00, 11.61s/it]


Generation complete: 266 reports saved
Combined output file: agent2_consulting_reports.csv
Individual reports saved in: reports/
